# Trabalho 6 — Sistemas Inteligentes
## Regressão Linear Múltipla — Dados de Imagem de Ultrassom Bovino

**Objetivo:** Prever o **PESO** do animal usando medidas morfométricas (colunas a partir de **AC**).

**Regras de implementação:**
- Funções principais implementadas **manualmente** (sem sklearn, sem numpy para cálculos)
- `pandas` utilizado **apenas** para leitura do CSV
- Divisão: **70% Treino / 30% Teste** com **seed = 50**
- Meta de precisão: **R² ≥ 0.8**

---
| Etapa | Módulo |
|---|---|
| 1 | Carregamento de dados |
| 2 | Filtragem de X e Y |
| 3 | Pré-processamento (Min-Max) |
| 4 | Divisão Treino/Teste |
| 5 | Regressão Linear (Equação Normal) |
| 6 | Cálculo do R² |
| 7 | Validação da Precisão |

import pandas as pd

def carregar_dados(caminho: str) -> pd.DataFrame:
    """
    Le o arquivo CSV e retorna um DataFrame bruto.
    pandas e usado APENAS para I/O; todos os calculos sao manuais.
    """
    df = pd.read_csv(caminho, encoding='utf-8')
    print(f'Registros carregados : {len(df)}')
    return df

# --- Execucao ---
CAMINHO_CSV = 'Dados_Projeto_Imagem_Ultrassom.xlsx.csv'
df_bruto = carregar_dados(CAMINHO_CSV)

In [ ]:
import pandas as pd

def carregar_dados(caminho: str) -> pd.DataFrame:
    """
    Lê o arquivo CSV e retorna um DataFrame bruto.
    pandas é usado APENAS para I/O; todos os cálculos são manuais.
    """
    df = pd.read_csv(caminho, encoding='utf-8')
    print(f'Registros carregados : {len(df)}')
    print(f'Colunas disponíveis  : {list(df.columns)}')
    return df

# --- Execução ---
CAMINHO_CSV = 'Dados_Projeto_Imagem_Ultrassom.xlsx.csv'
df_bruto = carregar_dados(CAMINHO_CSV)

def _converter_br_para_float(valor):
    """Converte '1,23' -> 1.23"""
    if isinstance(valor, str):
        return valor.replace(',', '.')
    return valor


def filtrar_xy(df: pd.DataFrame):
    """
    Y = PESO
    X = IDADE + medidas de ultrassom (AOL, COL, POL, RATIO, EGE, MOL, EC)
        + medidas morfometricas a partir de AC.
    Remove colunas com mais de 50 pct de NaN e linhas com NaN restante.
    Retorna: X (lista de listas), y (lista), nomes das features.
    """
    col_y = 'PESO'
    cols_candidatas_x = [
        'IDADE',
        'AOL (cm²)', 'COL (cm)', 'POL (cm)', 'RATIO (cm)',
        'EGE (mm)', 'MOL', 'EC',
        'AC', 'AG', 'CC', 'AP', 'P.C', 'CT', 'CO', 'CCAB',
        'LR', 'LCAB', 'LIL', 'LIS', 'Cga', 'Cper', 'PerPe', 'Ccau',
        'DC', 'CE'
    ]
    cols_x = [c for c in cols_candidatas_x if c in df.columns]
    subset = df[[col_y] + cols_x].copy()
    for col in subset.columns:
        subset[col] = subset[col].apply(_converter_br_para_float)
        subset[col] = pd.to_numeric(subset[col], errors='coerce')
    limite_nan = 0.5 * len(subset)
    cols_densas = [c for c in cols_x if subset[c].isna().sum() <= limite_nan]
    removidas = set(cols_x) - set(cols_densas)
    if removidas:
        print(f'Colunas removidas (>50% NaN): {removidas}')
    cols_x = cols_densas
    subset = subset[[col_y] + cols_x].dropna()
    print(f'Amostras apos limpeza : {len(subset)}')
    print(f'Features selecionadas : {cols_x}')
    y = subset[col_y].tolist()
    X = subset[cols_x].values.tolist()
    return X, y, cols_x


# --- Execucao ---
X_bruto, y, COLS_X = filtrar_xy(df_bruto)
print(f'\nPrimeiro valor y: {y[0]}')

In [ ]:
def _converter_br_para_float(valor):
    """Converte string no formato brasileiro ('1,23') para float."""
    if isinstance(valor, str):
        return valor.replace(',', '.')
    return valor


def filtrar_xy(df: pd.DataFrame):
    """
    Define Y = PESO e X = colunas a partir de AC.
    Remove colunas com mais de 50% de NaN e linhas com NaN restante.
    Retorna: X (lista de listas), y (lista), nomes das features.
    """
    col_y = 'PESO'
    cols_candidatas_x = [
        'AC', 'AG', 'CC', 'AP', 'P.C', 'CT', 'CO', 'CCAB',
        'LR', 'LCAB', 'LIL', 'LIS', 'Cga', 'Cper', 'PerPe', 'Ccau',
        'DC', 'CE'
    ]

    # Filtrar apenas colunas que existem no DataFrame
    cols_x = [c for c in cols_candidatas_x if c in df.columns]

    # Trabalhar apenas com as colunas necessárias
    subset = df[[col_y] + cols_x].copy()

    # Converter separador decimal BR → ponto
    for col in subset.columns:
        subset[col] = subset[col].apply(_converter_br_para_float)
        subset[col] = pd.to_numeric(subset[col], errors='coerce')

    # Remover colunas com mais de 50% de NaN
    limite_nan = 0.5 * len(subset)
    cols_densas = [c for c in cols_x if subset[c].isna().sum() <= limite_nan]
    removidas = set(cols_x) - set(cols_densas)
    if removidas:
        print(f'Colunas removidas (>50% NaN): {removidas}')
    cols_x = cols_densas

    # Remover linhas com NaN remanescente
    subset = subset[[col_y] + cols_x].dropna()

    print(f'Amostras após limpeza : {len(subset)}')
    print(f'Features selecionadas : {cols_x}')
    print(f'Variável alvo         : {col_y}')

    y = subset[col_y].tolist()
    X = subset[cols_x].values.tolist()
    return X, y, cols_x


# --- Execução ---
X_bruto, y, COLS_X = filtrar_xy(df_bruto)
print(f'\nExemplo — primeira amostra X: {X_bruto[0]}')
print(f'Exemplo — primeiro valor y  : {y[0]}')

---
## Etapa 3 — Pré-Processamento: Normalização Min-Max

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

Implementada **manualmente** (loop por coluna). Retorna os parâmetros de escala para uso futuro.

In [ ]:
def _min_lista(lst: list) -> float:
    """Calcula o mínimo de uma lista — implementado manualmente."""
    minimo = lst[0]
    for v in lst:
        if v < minimo:
            minimo = v
    return minimo


def _max_lista(lst: list) -> float:
    """Calcula o máximo de uma lista — implementado manualmente."""
    maximo = lst[0]
    for v in lst:
        if v > maximo:
            maximo = v
    return maximo


def normalizar_minmax(X: list):
    """
    Aplica Normalização Min-Max em cada feature.
    Retorna: X_normalizado, lista de mínimos, lista de máximos.
    """
    n_amostras = len(X)
    n_features = len(X[0])

    minimos, maximos = [], []
    for j in range(n_features):
        coluna = [X[i][j] for i in range(n_amostras)]
        minimos.append(_min_lista(coluna))
        maximos.append(_max_lista(coluna))

    X_norm = []
    for i in range(n_amostras):
        linha = []
        for j in range(n_features):
            denom = maximos[j] - minimos[j]
            if denom == 0.0:
                linha.append(0.0)
            else:
                linha.append((X[i][j] - minimos[j]) / denom)
        X_norm.append(linha)

    print(f'Normalização concluída: {n_features} features | {n_amostras} amostras')
    print(f'Exemplo — primeira amostra normalizada: {[round(v,4) for v in X_norm[0]]}')
    return X_norm, minimos, maximos


# --- Execução ---
X_norm, MINIMOS, MAXIMOS = normalizar_minmax(X_bruto)

---
## Etapa 4 — Divisão Treino/Teste (70/30, seed=50)

O embaralhamento é feito com um **Gerador Congruencial Linear (LCG)** implementado manualmente — sem uso de `random` do Python para a lógica principal.

In [ ]:
def _embaralhar_lcg(n: int, seed: int = 50) -> list:
    """
    Gera uma permutação aleatória de índices 0..n-1 usando um
    Gerador Congruencial Linear (LCG) implementado manualmente.
    Parâmetros de Knuth: a=1664525, c=1013904223, m=2^32.
    """
    a, c, m = 1_664_525, 1_013_904_223, 2 ** 32
    estado = seed
    indices = list(range(n))

    for i in range(n - 1, 0, -1):          # Fisher-Yates shuffle
        estado = (a * estado + c) % m
        j = estado % (i + 1)
        indices[i], indices[j] = indices[j], indices[i]

    return indices


def dividir_treino_teste(X: list, y: list,
                         razao_treino: float = 0.7,
                         seed: int = 50):
    """
    Embaralha com seed fixo e divide os dados em treino e teste.
    Retorna: X_treino, y_treino, X_teste, y_teste.
    """
    indices = _embaralhar_lcg(len(X), seed)
    X_emb = [X[i] for i in indices]
    y_emb = [y[i] for i in indices]

    n_treino = int(len(X) * razao_treino)
    X_treino = X_emb[:n_treino]
    y_treino = y_emb[:n_treino]
    X_teste  = X_emb[n_treino:]
    y_teste  = y_emb[n_treino:]

    print(f'Total de amostras : {len(X)}')
    print(f'Treino (70%)      : {len(y_treino)} amostras')
    print(f'Teste  (30%)      : {len(y_teste)} amostras')
    return X_treino, y_treino, X_teste, y_teste


# --- Execução ---
X_treino, y_treino, X_teste, y_teste = dividir_treino_teste(
    X_norm, y, razao_treino=0.7, seed=50
)

# --- Operacoes de Algebra Linear (implementacao manual) ---

def _adicionar_bias(X):
    """Adiciona coluna de 1s (termo independente b0)."""
    return [[1.0] + row[:] for row in X]

def _transpor(M):
    """Transpoe matriz M."""
    n_lin, n_col = len(M), len(M[0])
    return [[M[i][j] for i in range(n_lin)] for j in range(n_col)]

def _mult_mat(A, B):
    """Multiplicacao de matrizes A x B - manual."""
    n, m, p = len(A), len(B[0]), len(B)
    C = [[0.0]*m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            s = 0.0
            for k in range(p):
                s += A[i][k] * B[k][j]
            C[i][j] = s
    return C

def _inverter_gauss(M):
    """
    Inversao por Eliminacao de Gauss-Jordan com pivotamento parcial.
    Implementada 100% manualmente.
    """
    n = len(M)
    aug = [M[i][:] + [1.0 if i==j else 0.0 for j in range(n)] for i in range(n)]
    for col in range(n):
        max_row = max(range(col, n), key=lambda r: abs(aug[r][col]))
        aug[col], aug[max_row] = aug[max_row], aug[col]
        pivo = aug[col][col]
        if abs(pivo) < 1e-12:
            raise ValueError('Matriz singular.')
        for j in range(2*n):
            aug[col][j] /= pivo
        for row in range(n):
            if row != col:
                f = aug[row][col]
                for j in range(2*n):
                    aug[row][j] -= f * aug[col][j]
    return [aug[i][n:] for i in range(n)]

def _mult_mat_vec(M, v):
    """Multiplica matriz M por vetor v - manual."""
    return [sum(M[i][j]*v[j] for j in range(len(v))) for i in range(len(M))]

def _regularizar_XtX(XtX, lam=1e-4):
    """
    Regularizacao de Tikhonov (Ridge): XtX_reg = XtX + lambda*I
    Evita singularidade quando features sao correlacionadas. Manual.
    """
    n = len(XtX)
    reg = [row[:] for row in XtX]
    for i in range(n):
        reg[i][i] += lam
    return reg


def regressao_linear(X_treino, y_treino, lam=1e-4):
    """
    Equacao Normal com regularizacao Ridge:
        beta = (XtX + lambda*I)^-1 * Xty
    Algebra linear 100% manual.
    """
    X_b     = _adicionar_bias(X_treino)
    Xt      = _transpor(X_b)
    XtX     = _mult_mat(Xt, X_b)
    XtX_reg = _regularizar_XtX(XtX, lam)
    XtXi    = _inverter_gauss(XtX_reg)
    Xty     = _mult_mat_vec(Xt, y_treino)
    beta    = _mult_mat_vec(XtXi, Xty)
    print(f'Modelo treinado: {len(beta)} coef (b0 + {len(beta)-1} pesos) | lambda={lam}')
    return beta

def prever(X, beta):
    """Predicoes y_hat = X * beta."""
    X_b = _adicionar_bias(X)
    return [sum(X_b[i][j]*beta[j] for j in range(len(beta))) for i in range(len(X_b))]


# --- Execucao ---
BETA = regressao_linear(X_treino, y_treino)

In [ ]:
# --- Operações de Álgebra Linear (implementação manual) ---

def _adicionar_bias(X: list) -> list:
    """Adiciona coluna de 1s à esquerda (termo independente β₀)."""
    return [[1.0] + row[:] for row in X]


def _transpor(M: list) -> list:
    """Transpõe uma matriz M (lista de listas)."""
    n_lin, n_col = len(M), len(M[0])
    return [[M[i][j] for i in range(n_lin)] for j in range(n_col)]


def _mult_mat(A: list, B: list) -> list:
    """Multiplicação de matrizes A × B — implementada manualmente."""
    n, m, p = len(A), len(B[0]), len(B)
    C = [[0.0] * m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            soma = 0.0
            for k in range(p):
                soma += A[i][k] * B[k][j]
            C[i][j] = soma
    return C


def _inverter_gauss(M: list) -> list:
    """
    Inverte a matriz M usando Eliminação de Gauss-Jordan
    com pivotamento parcial — implementada 100% manualmente.
    """
    n = len(M)
    # Matriz aumentada [M | I]
    aug = [
        M[i][:] + [1.0 if i == j else 0.0 for j in range(n)]
        for i in range(n)
    ]

    for col in range(n):
        # Pivotamento parcial: escolhe a linha com maior valor absoluto na coluna
        max_row = max(range(col, n), key=lambda r: abs(aug[r][col]))
        aug[col], aug[max_row] = aug[max_row], aug[col]

        pivo = aug[col][col]
        if abs(pivo) < 1e-12:
            raise ValueError('Matriz singular — não é possível inverter.')

        # Normalizar linha do pivô
        for j in range(2 * n):
            aug[col][j] /= pivo

        # Zerar todas as outras linhas nessa coluna
        for row in range(n):
            if row != col:
                fator = aug[row][col]
                for j in range(2 * n):
                    aug[row][j] -= fator * aug[col][j]

    return [aug[i][n:] for i in range(n)]


def _mult_mat_vec(M: list, v: list) -> list:
    """Multiplica matriz M por vetor v — implementada manualmente."""
    return [sum(M[i][j] * v[j] for j in range(len(v))) for i in range(len(M))]


# --- Regressão Linear ---

def regressao_linear(X_treino: list, y_treino: list) -> list:
    """
    Calcula β pela Equação Normal: β = (XᵀX)⁻¹ · Xᵀy
    Toda a álgebra linear é implementada manualmente.
    """
    X_b  = _adicionar_bias(X_treino)      # Adiciona coluna de 1s
    Xt   = _transpor(X_b)                 # Xᵀ
    XtX  = _mult_mat(Xt, X_b)            # XᵀX
    XtXi = _inverter_gauss(XtX)          # (XᵀX)⁻¹  ← Gauss-Jordan manual
    Xty  = _mult_mat_vec(Xt, y_treino)   # Xᵀy
    beta = _mult_mat_vec(XtXi, Xty)      # β
    print(f'Modelo treinado: {len(beta)} coeficientes (β₀ + {len(beta)-1} pesos)')
    return beta


def prever(X: list, beta: list) -> list:
    """Gera predições ŷ = Xβ para um conjunto de amostras X."""
    X_b = _adicionar_bias(X)
    return [
        sum(X_b[i][j] * beta[j] for j in range(len(beta)))
        for i in range(len(X_b))
    ]


# --- Execução ---
BETA = regressao_linear(X_treino, y_treino)

---
## Etapa 6 — Cálculo do R²

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

Implementado **manualmente** com loops puros.

In [ ]:
def _media(valores: list) -> float:
    """Calcula a média aritmética — implementada manualmente."""
    return sum(valores) / len(valores)


def calcular_r2(y_real: list, y_pred: list) -> float:
    """
    Calcula o Coeficiente de Determinação R² manualmente:
        SS_res = Σ(yᵢ - ŷᵢ)²
        SS_tot = Σ(yᵢ - ȳ)²
        R²     = 1 - SS_res / SS_tot
    """
    media_y = _media(y_real)
    ss_res = sum((y_real[i] - y_pred[i]) ** 2 for i in range(len(y_real)))
    ss_tot = sum((y_real[i] - media_y) ** 2 for i in range(len(y_real)))
    if ss_tot == 0:
        return 0.0
    return 1.0 - (ss_res / ss_tot)


# --- Execução ---
y_pred_treino = prever(X_treino, BETA)
y_pred_teste  = prever(X_teste,  BETA)

R2_TREINO = calcular_r2(y_treino, y_pred_treino)
R2_TESTE  = calcular_r2(y_teste,  y_pred_teste)

print(f'R² no conjunto de TREINO : {R2_TREINO:.4f}')
print(f'R² no conjunto de TESTE  : {R2_TESTE:.4f}')

---
## Etapa 7 — Validação da Precisão

Verifica se o R² no conjunto de **teste** atinge o limiar mínimo de **0.8**.

In [ ]:
def validar_precisao(r2: float, limiar: float = 0.8) -> tuple:
    """
    Verifica se o R² atinge o limiar mínimo de precisão.
    Retorna (bool aprovado, str status).
    """
    aprovado = r2 >= limiar
    status = '✅ APROVADO' if aprovado else '❌ REPROVADO'
    return aprovado, status


# --- Execução ---
APROVADO, STATUS = validar_precisao(R2_TESTE, limiar=0.8)

print('=' * 50)
print('  RESULTADO DA VALIDAÇÃO')
print('=' * 50)
print(f'  R² Teste   : {R2_TESTE:.4f}')
print(f'  Limiar     : 0.8000')
print(f'  Resultado  : {STATUS}')
print('=' * 50)

---
## Resumo Final

In [ ]:
print('=' * 55)
print('  RESUMO COMPLETO DO PIPELINE')
print('=' * 55)
print(f'  Features usadas  : {COLS_X}')
print(f'  Total amostras   : {len(y)}')
print(f'  Amostras treino  : {len(y_treino)}')
print(f'  Amostras teste   : {len(y_teste)}')
print(f'  R² Treino        : {R2_TREINO:.4f}')
print(f'  R² Teste         : {R2_TESTE:.4f}')
print(f'  Precisão (≥0.8)  : {STATUS}')
print('=' * 55)

# Tabela: Real vs Predito
print(f'\n  Predições vs Valores Reais (primeiros 15 do teste):')
print(f'  {"#":>4} | {"Real":>8} | {"Predito":>10} | {"Erro Abs":>10}')
print('  ' + '-' * 42)
for i in range(min(15, len(y_teste))):
    erro = abs(y_teste[i] - y_pred_teste[i])
    print(f'  {i+1:>4} | {y_teste[i]:>8.2f} | {y_pred_teste[i]:>10.2f} | {erro:>10.2f}')

# Coeficientes
print(f'\n  Coeficientes β do modelo:')
print(f'  {"β₀ (bias)":>12} = {BETA[0]:.4f}')
for i, col in enumerate(COLS_X):
    print(f'  {col:>12} = {BETA[i+1]:.4f}')